In [1]:
%matplotlib inline

import os
import sys

sys.path.append('../../../../')

import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [2]:
from __future__ import annotations
from typing import Optional, Union, Sequence

import torch
from torch.nn.modules.utils import _pair

import numpy as np

%load_ext autoreload
%autoreload 2

from computer_vision.slowfast.mmaction.datasets.transforms.loading import DecordInit, SampleFrames, DecordDecode
from computer_vision.slowfast.mmaction.datasets.transforms.processing import Resize, _init_lazy_of_proper, RandomCrop, CenterCrop, ThreeCrop, \
RandomResizedCrop, Flip
from computer_vision.slowfast.mmaction.datasets.transforms.formatting import FormatShape, PackActionInputs
from computer_vision.slowfast.mmengine.utils.misc import is_tuple_of
from computer_vision.slowfast.mmcv.image.geometric import imflip
from computer_vision.slowfast.mmcv.image.photometric import iminvert
from computer_vision.slowfast.mmaction.evaluation.metrics.acc_metric import to_tensor
from computer_vision.slowfast.mmaction.structures.action_data_sample import ActionDataSample
from computer_vision.slowfast.mmengine.structures.instance_data import InstanceData

In [3]:
output_dirpath='D:/results/ucf101'
video_fpath=f'{output_dirpath}/demo.mp4'
assert os.path.isfile(video_fpath)

data=dict(filename=video_fpath, label=-1, start_index=0, modality='RGB')
print(f"{data=}")

# Create Decord
arguments={'type': 'DecordInit', 'io_backend': 'disk'}
arguments.pop('type')
decordinit=DecordInit(**arguments)
results=decordinit(data)
print(f'Decord-init: {results.keys()=}')

# Sample frames
arguments={'type': 'SampleFrames', 'clip_len': 32, 'frame_interval': 2, 'num_clips': 10, 'test_mode': True} #, 'target_fps':30}
arguments.pop('type')
sample_frames_op=SampleFrames(**arguments)
results=sample_frames_op(results)
print(f'Sample frames: {results.keys()=}')

decord_decode=DecordDecode()
results=decord_decode(results)
print(f'Decode: {results.keys()=}')

resize_op=Resize(scale=(-1, 256), keep_ratio=True, interpolation='bilinear', lazy=False) 
results=resize_op(results)
print(f'Resize: {results.keys()=}')
print(f"{type(results['imgs'])=}, {[x.shape for i, x in enumerate(results['imgs']) if i<4]}, {len(results['imgs'])=}")

rnd_resize_op=RandomResizedCrop()
results=rnd_resize_op(results)
print(f'ThreeCrop: {results.keys()=}')
print(f"{type(results['imgs'])=}, {[x.shape for i, x in enumerate(results['imgs']) if i<4]}, {len(results['imgs'])=}")

resize_op=Resize(scale=(224, 224), keep_ratio=False, interpolation='bilinear', lazy=False) 
results=resize_op(results)
print(f'Resize: {results.keys()=}')
print(f"{type(results['imgs'])=}, {[x.shape for i, x in enumerate(results['imgs']) if i<4]}, {len(results['imgs'])=}")
print(f"image size {[x.shape for i, x in enumerate(results['imgs']) if i < 7]}")

flip_op=Flip(flip_ratio=1.)
results=flip_op(results)
print(f'Flip: {results.keys()=}')
print(f"{type(results['imgs'])=}, {[x.shape for i, x in enumerate(results['imgs']) if i<4]}, {len(results['imgs'])=}")
print(f"image size {[x.shape for i, x in enumerate(results['imgs']) if i < 7]}")

format_op=FormatShape(input_format='NCTHW')
results=format_op(results)
print(f'FormatShape: {results.keys()=}')
print(f"{results['imgs'].shape=}, {results['imgs'].dtype}")

pack_op=PackActionInputs()
outputs=pack_op(results)

data={'filename': 'D:/results/ucf101/demo.mp4', 'label': -1, 'start_index': 0, 'modality': 'RGB'}
Decord-init: results.keys()=dict_keys(['filename', 'label', 'start_index', 'modality', 'total_frames', 'video_reader', 'avg_fps'])
Sample frames: results.keys()=dict_keys(['filename', 'label', 'start_index', 'modality', 'total_frames', 'video_reader', 'avg_fps', 'frame_inds', 'clip_len', 'frame_interval', 'num_clips'])
Decode: results.keys()=dict_keys(['filename', 'label', 'start_index', 'modality', 'total_frames', 'video_reader', 'avg_fps', 'frame_inds', 'clip_len', 'frame_interval', 'num_clips', 'imgs', 'original_shape', 'img_shape'])
Resize: results.keys()=dict_keys(['filename', 'label', 'start_index', 'modality', 'total_frames', 'video_reader', 'avg_fps', 'frame_inds', 'clip_len', 'frame_interval', 'num_clips', 'imgs', 'original_shape', 'img_shape', 'scale_factor', 'keep_ratio'])
type(results['imgs'])=<class 'list'>, [(256, 340, 3), (256, 340, 3), (256, 340, 3), (256, 340, 3)], len(res

In [10]:
print(f"{outputs.keys()=}")
print(f"{type(outputs['inputs'])=}, {outputs['inputs'].dtype=}, {outputs['inputs'].shape=}")
print("outputs['data_samples']=", outputs['data_samples'])
print(f"{outputs['data_samples'].get('gt_label')=}, {outputs['data_samples'].get('img_shape')=}")

outputs.keys()=dict_keys(['inputs', 'data_samples'])
type(outputs['inputs'])=<class 'torch.Tensor'>, outputs['inputs'].dtype=torch.uint8, outputs['inputs'].shape=torch.Size([10, 3, 32, 224, 224])
outputs['data_samples']= <ActionDataSample(

    META INFORMATION
    img_shape:(224, 224)

    DATA FIELDS
    gt_label:tensor([-1])
) at 0x209c38d0460>
outputs['data_samples'].get('gt_label')=tensor([-1]), outputs['data_samples'].get('img_shape')=(224, 224)
